# E-Commerce Lakehouse: Silver Payments

## Standardize payment fields and data types

Preserve order identifiers, standardize payment type text, and
convert payment sequence and installment counts to integers.
Convert payment_value to a decimal with two decimal places.
Validate the results before saving.

In [0]:
from pyspark.sql.functions import col, trim, lower

bronze_payments_df = spark.table(
    "workspace.ecommerce.bronze_payments"
)

silver_payments_df = (
    bronze_payments_df
    .withColumn("order_id", trim(col("order_id")))
    .withColumn("payment_type", lower(trim(col("payment_type"))))
    .withColumn(
        "payment_sequential",
        col("payment_sequential").cast("int")
    )
    .withColumn(
        "payment_installments",
        col("payment_installments").cast("int")
    )
    .withColumn(
        "payment_value",
        col("payment_value").cast("decimal(12,2)")
    )
)

silver_payments_df.printSchema()
display(silver_payments_df.limit(5))

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: decimal(12,2) (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)



order_id,payment_sequential,payment_type,payment_installments,payment_value,ingestion_timestamp
b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33,2026-10-17T18:25:24.448Z
a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39,2026-10-17T18:25:24.448Z
25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71,2026-10-17T18:25:24.448Z
ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78,2026-10-17T18:25:24.448Z
42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45,2026-10-17T18:25:24.448Z


## Validate Silver payments

Compare row counts and null counts before and after conversion.
Check required fields, payment key uniqueness, numeric values,
payment types, and links to Silver orders.

Inspect zero-value payments and zero installments without
automatically treating them as errors.

In [0]:
from pyspark.sql import functions as F

converted_columns = [
    "payment_sequential",
    "payment_installments",
    "payment_value"
]

def null_summary(df):
    return df.agg(
        F.count("*").alias("rows"),
        *[
            F.count(
                F.when(F.col(name).isNull(), 1)
            ).alias(f"{name}_nulls")
            for name in converted_columns
        ]
    )

print("Bronze summary:")
null_summary(bronze_payments_df).show()

print("Silver summary:")
null_summary(silver_payments_df).show()

silver_payments_df.agg(
    F.count(F.when(
        F.col("order_id").isNull()
        | (F.trim(F.col("order_id")) == ""), 1
    )).alias("missing_order_ids"),

    F.count(F.when(
        F.col("payment_type").isNull()
        | (F.trim(F.col("payment_type")) == ""), 1
    )).alias("missing_payment_types"),

    F.count(F.when(
        F.col("payment_sequential") <= 0, 1
    )).alias("nonpositive_sequences"),

    F.count(F.when(
        F.col("payment_installments") < 0, 1
    )).alias("negative_installments"),

    F.count(F.when(
        F.col("payment_value") < 0, 1
    )).alias("negative_payment_values"),

    F.count(F.when(
        F.col("payment_installments") == 0, 1
    )).alias("zero_installments"),

    F.count(F.when(
        F.col("payment_value") == 0, 1
    )).alias("zero_payment_values")
).show(truncate=False)

duplicate_keys = (
    silver_payments_df
    .groupBy("order_id", "payment_sequential")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Duplicate payment keys:", duplicate_keys)

silver_payments_df.groupBy("payment_type").count().show()

order_ids_df = spark.table(
    "workspace.ecommerce.silver_orders"
).select("order_id")

unmatched_count = silver_payments_df.join(
    order_ids_df,
    on="order_id",
    how="left_anti"
).count()

print("Payments without a matching order:", unmatched_count)

Bronze summary:
+------+------------------------+--------------------------+-------------------+
|  rows|payment_sequential_nulls|payment_installments_nulls|payment_value_nulls|
+------+------------------------+--------------------------+-------------------+
|103886|                       0|                         0|                  0|
+------+------------------------+--------------------------+-------------------+

Silver summary:
+------+------------------------+--------------------------+-------------------+
|  rows|payment_sequential_nulls|payment_installments_nulls|payment_value_nulls|
+------+------------------------+--------------------------+-------------------+
|103886|                       0|                         0|                  0|
+------+------------------------+--------------------------+-------------------+

+-----------------+---------------------+---------------------+---------------------+-----------------------+-----------------+-------------------+
|missing

## Inspect unusual payment records

Inspect payments with zero installments, zero payment value,
or an undefined payment type. Preserve these records while
determining how they should be treated in analysis.

In [0]:
unusual_payments_df = silver_payments_df.filter(
    (F.col("payment_installments") == 0)
    | (F.col("payment_value") == 0)
    | (F.col("payment_type") == "not_defined")
)

display(
    unusual_payments_df.select(
        "order_id",
        "payment_sequential",
        "payment_type",
        "payment_installments",
        "payment_value"
    ).orderBy("payment_type", "order_id", "payment_sequential")
)

order_id,payment_sequential,payment_type,payment_installments,payment_value
1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94
744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.00
4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.00
c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.00
45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.00
6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.00
8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.00
b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.00
fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.00


In [0]:
display(
    unusual_payments_df.select(
        "order_id",
        "payment_sequential",
        "payment_type",
        "payment_installments",
        "payment_value"
    ).orderBy("payment_type", "order_id", "payment_sequential")
)

order_id,payment_sequential,payment_type,payment_installments,payment_value
1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94
744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.00
4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.00
c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.00
45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.00
6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.00
8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.00
b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.00
fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.00


## Flag unusual payments and refresh Silver

Preserve all payment records and flag zero installments, zero
payment values, and undefined payment types.

Validate payment keys and required numeric fields before replacing
the Silver table. Verify saved row and flag counts after writing.

In [0]:
from pyspark.sql import functions as F

silver_payments_df = (
    silver_payments_df
    .withColumn(
        "zero_installments_flag",
        F.col("payment_installments") == 0
    )
    .withColumn(
        "zero_payment_value_flag",
        F.col("payment_value") == 0
    )
    .withColumn(
        "undefined_payment_type_flag",
        F.col("payment_type") == "not_defined"
    )
)

invalid_count = silver_payments_df.filter(
    F.col("order_id").isNull()
    | (F.trim(F.col("order_id")) == "")
    | F.col("payment_type").isNull()
    | (F.trim(F.col("payment_type")) == "")
    | F.col("payment_sequential").isNull()
    | (F.col("payment_sequential") <= 0)
    | F.col("payment_installments").isNull()
    | (F.col("payment_installments") < 0)
    | F.col("payment_value").isNull()
    | (F.col("payment_value") < 0)
).count()

duplicate_count = (
    silver_payments_df
    .groupBy("order_id", "payment_sequential")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

row_count = silver_payments_df.count()

if (
    row_count != 103886
    or invalid_count != 0
    or duplicate_count != 0
):
    raise ValueError(
        "Silver payment validation failed. "
        "Stopped before replacing the table."
    )

table_name = "workspace.ecommerce.silver_payments"

(
    silver_payments_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(table_name)
)

saved_payments_df = spark.table(table_name)

saved_checks = saved_payments_df.agg(
    F.count("*").alias("saved_rows"),
    F.count(F.when(
        F.col("zero_installments_flag"), 1
    )).alias("zero_installments"),
    F.count(F.when(
        F.col("zero_payment_value_flag"), 1
    )).alias("zero_payment_values"),
    F.count(F.when(
        F.col("undefined_payment_type_flag"), 1
    )).alias("undefined_payment_types")
).first()

if (
    saved_checks["saved_rows"] != row_count
    or saved_checks["zero_installments"] != 2
    or saved_checks["zero_payment_values"] != 9
    or saved_checks["undefined_payment_types"] != 3
):
    raise ValueError("Saved payment counts differ from the expected dataset.")

print("Silver payments refreshed:", saved_checks["saved_rows"])
print("Zero installments:", saved_checks["zero_installments"])
print("Zero payment values:", saved_checks["zero_payment_values"])
print("Undefined payment types:", saved_checks["undefined_payment_types"])

Silver payments table created.
+----------+-----------------+-------------------+-----------------------+
|saved_rows|zero_installments|zero_payment_values|undefined_payment_types|
+----------+-----------------+-------------------+-----------------------+
|    103886|                2|                  9|                      3|
+----------+-----------------+-------------------+-----------------------+

